# nanoHUB-QE Tutorial

This notebook shows how to use `nanohubqe` on nanoHUB, including loading Quantum ESPRESSO with the `use` command behavior.

In [ ]:
import nanohubqe
from nanohubqe import __version__
print('nanohubqe version:', __version__)

## 1. Load Quantum ESPRESSO on nanoHUB

On nanoHUB, load the QE module before running workflows.
Use the Python function-based interface below.

In [ ]:
from nanohubqe import list_available_modules, load_quantum_espresso

qe_like = list_available_modules('qe')
espresso_like = list_available_modules('espresso')
print('QE-like modules:', qe_like[:10])
print('ESPRESSO-like modules:', espresso_like[:10])

try:
    loaded = load_quantum_espresso()
    print('Loaded module:', loaded)
except Exception as exc:
    print('Auto-detect failed:', exc)
    print("Call load_quantum_espresso('your-module-name') if needed.")


## 2. Build a nanoHUB-style Si SCF + DOS + Bands workflow

In [ ]:
from pathlib import Path
from shutil import which
from nanohubqe import silicon_bands_dos_reference_workflow

sim = silicon_bands_dos_reference_workflow(
    a=5.43,
    ecutwfc=16.0,
    ecutrho=96.0,
    nbnd=8,
    scf_k_points=(4, 4, 4, 0, 0, 0),
    dos_emin=-6.0,
    dos_emax=10.0,
    dos_deltae=0.1,
    include_plotband=False,
    pseudo_dir='./pseudo',
    pseudo_file='Si.UPF',
)

workdir = Path('runs/tutorial-si')
workdir.mkdir(parents=True, exist_ok=True)

pseudo_ready = True
try:
    pseudo_status = sim.prepare_pseudopotentials(workdir=workdir)
    for item in pseudo_status:
        print(f'{item.pseudo_file}: {item.action} -> {item.target_path}')
except FileNotFoundError as exc:
    pseudo_ready = False
    print('Pseudo provisioning failed:', exc)
    print('Set ESPRESSO_PSEUDO or pass source_urls to sim.prepare_pseudopotentials().')

required = ['pw.x', 'dos.x', 'bands.x']
missing = [exe for exe in required if which(exe) is None]
dry_run = bool(missing)
if not pseudo_ready:
    dry_run = True

if dry_run:
    if missing:
        print('Missing executables:', ', '.join(missing))
    print('Falling back to dry_run=True (commands only).')
else:
    print('Running workflow with real QE executables...')

sim.run(workdir=workdir, dry_run=dry_run)
for step_name, result in sim.results.items():
    print(step_name, 'rc=', result.returncode, 'ok=', result.ok)

if 'scf' in sim.results and not sim.step_result('scf').ok:
    scf_log = sim.step_result('scf').output_file
    print('SCF failed. Log file:', scf_log)
    tail = Path(scf_log).read_text(encoding='utf-8', errors='ignore').splitlines()[-40:]
    print('---- SCF log tail ----')
    print('\n'.join(tail))


## 3. Inspect expected/discovered outputs and run records

In [ ]:
print('Executed steps:', list(sim.results))

try:
    dos_result = sim.step_result('dos')
    print('DOS return code:', dos_result.returncode)
    print('DOS discovered outputs:', [p.name for p in dos_result.discovered_outputs])
except KeyError:
    print('DOS step not available (workflow may have stopped early).')


## 4. Plot outputs

These helpers use files generated by the latest `sim.run(...)` call.


In [ ]:
from IPython.display import display

display(sim.plot_total_energy(backend='plotly'))
display(sim.plot_dos(backend='plotly'))
display(sim.plot_bands(backend='plotly'))


## 5. Remote execution with submit

This executes each step through HUBzero `submit`, waits for completion, and attempts to sync outputs back to the local workdir for plotting.



In [ ]:
from nanohubqe import QERunner, SubmitConfig, silicon_bands_dos_reference_workflow

remote_sim = silicon_bands_dos_reference_workflow(
    a=5.43,
    ecutwfc=16.0,
    ecutrho=96.0,
    nbnd=8,
    scf_k_points=(4, 4, 4, 0, 0, 0),
    dos_emin=-6.0,
    dos_emax=10.0,
    dos_deltae=0.1,
    include_plotband=False,
    pseudo_dir='./pseudo',
    pseudo_file='Si.UPF',
)

remote_workdir = Path('runs/tutorial-si-remote')
remote_workdir.mkdir(parents=True, exist_ok=True)
remote_sim.prepare_pseudopotentials(workdir=remote_workdir)

submit_cfg = SubmitConfig(
    venue='nanohub',
    nodes=4,
    walltime='00:30:00',
    manager='espresso-6.8_mpi-cleanup_pw',
    run_name='si-reference-remote',
    executable_prefix='espresso-7.1',
)

remote_runner = QERunner(default_backend='submit')

remote_sim.run_submit(
    workdir=remote_workdir,
    runner=remote_runner,
    submit_config=submit_cfg,
    dry_run=False,
    wait=True,
    sync_outputs=True,
    poll_interval=20.0,
    wait_timeout=3600.0,
)

for step_name, result in remote_sim.results.items():
    print(step_name, 'rc=', result.returncode, 'status=', result.remote_status, 'synced=', result.outputs_synced)

